# Sistema para apuração de fiscal - Empresa vs Contabilidade

## Bibliotecas

In [119]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

## 1.0 Dados Empresa

### 1.1 Coleta de dados

In [ ]:
df_empresa = pd.read_excel("./dataset/livro_empresa.xlsx", header=1)

### 1.2 Alterar o nome das colunas

In [121]:
df_empresa = df_empresa.rename(columns={
    'Número': 'nf_empresa',
    'Tributado': 'vl_icms_empresa',
    'Tributado.1': 'vl_ipi_empresa',
    'Nat. Op.': 'cfop_empresa',
    'Valor Contábil': 'vl_cont_empresa'
})

### 1.3 Remoção, limpeza e formatação dos dados

#### 1.3.1 Remover colunas desnecessárias

In [122]:
delete_columns_empresa = ['Espécie', 
                          'Série', 
                          'Emissão', 
                          'Entrada', 
                          'Cód Emitente', 
                          'UF', 
                          'Incidência',
                          '%',
                          'Base Cálc.',
                          'Isento',
                          'Outros',
                          'Incidência.1',
                          '%.1',
                          'Base Cálc..1',
                          'Observação Substituição Tributária',
                          'Isento.1',
                          'Outros.1',
                          'Observação']
df_empresa = df_empresa.drop(columns=delete_columns_empresa)

#### 1.3.2 Remover linhas em branco

In [123]:
df_empresa = df_empresa.dropna()

#### 1.3.3 Formatação dos dados

In [124]:
df_empresa['nf_empresa'] = df_empresa['nf_empresa'].astype('int')
df_empresa['cfop_empresa'] = df_empresa['cfop_empresa'].astype('int')

#### 1.3.4 Criando a coluna ID unico

In [125]:
df_empresa['id_unico_empresa'] = df_empresa['nf_empresa'].astype('str') + '|' + df_empresa['cfop_empresa'].astype('str')

#### 1.3.5 Somando os valores cujo os ID's unicos sejam iguais

In [126]:
df_empresa = df_empresa.groupby('id_unico_empresa', as_index=False).sum(numeric_only=True)

#### 1.3.6 Preenchendo os valores em branco com 0

In [127]:
df_empresa = df_empresa.fillna(0)

#### 1.3.7 Gerando os totais dos dados da empresa

In [128]:
qtde_nf_empresa = (df_empresa['nf_empresa']).dropna().drop_duplicates().count()
valor_contabil_total_empresa = df_empresa['vl_cont_empresa'].sum()
valor_icms_total_empresa = df_empresa['vl_icms_empresa'].sum()
valor_ipi_total_empresa = df_empresa['vl_ipi_empresa'].sum()

## 2.0 Dados contabilidade (ICMS)

### 2.1 Coleta de dados

In [ ]:
df_contabilidade_icms = pd.read_excel("./dataset/livro_icms_contabilidade.xlsx", header=6)


### 2.2 Alteraro nome das colunas

In [130]:
df_contabilidade_icms = df_contabilidade_icms.rename(columns={
    'Número': 'nf_icms_contabilidade',
    'Natureza': 'cfop_icms',
    "Valor ICMS": 'vl_icms_contabilidade',
    'Vlr Contábil': 'vl_cont_icms_contabilidade'
})

### 2.3 Remoção, limpeza e formatação dos dados

#### 2.3.1 Removendo colunas desnecessárias

In [131]:
df_contabilidade_icms = df_contabilidade_icms.drop(columns=['Nr Lcto', 
                                          'Pessoa', 
                                          'Data Lcto', 
                                          'Esp', 
                                          'Série',
                                          'Base ICMS', 
                                          'Isentas ICMS', 
                                          'Outras ICMS'])

#### 2.3.2 Removendo as linhas em branco

In [132]:
df_contabilidade_icms = df_contabilidade_icms.dropna()

#### 2.3.3 Limpeza dos dados onde apareça a palavra 'Natureza'

In [133]:
df_contabilidade_icms = df_contabilidade_icms[df_contabilidade_icms['cfop_icms'] != 'Natureza']

#### 2.3.4 Formatação dos dados

In [134]:

colunas_icms_float = ['vl_icms_contabilidade', 'vl_cont_icms_contabilidade']
colunas_icms_int = ['cfop_icms','nf_icms_contabilidade']
for col in colunas_icms_int:
    df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('int')

for col in colunas_icms_float:
    df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('float')

#### 2.3.5 Separando apenas os CFOP's menores ou iguais a 4000

In [135]:
filtro = df_contabilidade_icms.query('cfop_icms <= 4000')
df_contabilidade_icms = filtro


#### 2.3.6 Criando a coluna ID unico

In [136]:
df_contabilidade_icms['id_unico_ICMS'] = df_contabilidade_icms['nf_icms_contabilidade'].astype('str') + "|" + df_contabilidade_icms['cfop_icms'].astype('str')

#### 2.3.7 Gerando os totais dos dados da contabilidade ICMS

In [137]:
qtde_nf_icms_contabilidade = (df_contabilidade_icms['nf_icms_contabilidade']).dropna().drop_duplicates().count()
valor_contabil_total_icms_contabilidade = df_contabilidade_icms['vl_cont_icms_contabilidade'].sum()
valor_icms_total_contabilidade = df_contabilidade_icms['vl_icms_contabilidade'].sum()

## 3.0 Dados contabilidade (IPI)

### 3.1 Coleta de dados

In [ ]:
df_contabilidade_ipi = pd.read_excel("./dataset/livro_ipi_contabilidade.xlsx", header=6)

### 3.2 Alterar o nome das colunas

In [139]:
df_contabilidade_ipi = df_contabilidade_ipi.rename(columns={
    'Número': 'nf_ipi_contabilidade',
    'Natureza': 'cfop_ipi',
    'Valor IPI': 'vl_ipi_contabilidade',
    'Vlr Contabil': 'vl_cont_ipi_contabilidade'
})

### 3.3 Remoção, limpeza e formatação dos dados

#### 3.3.1 Removendo colunas desnecessárias

In [140]:
df_contabilidade_ipi = df_contabilidade_ipi.drop(columns=['Nr Lcto', 
                                        'Fornecedor', 
                                        'Data Lcto', 
                                        'Esp', 
                                        'Série',
                                        'Base IPI', 
                                        'Isentas IPI', 
                                        'Outras IPI'])

#### 3.3.2 Removendo as linhas em branco

In [141]:
df_contabilidade_ipi = df_contabilidade_ipi.dropna()

#### 3.3.3 Limpeza dos dados onde apareça a palavra 'Natureza'

In [142]:
df_contabilidade_ipi = df_contabilidade_ipi[df_contabilidade_ipi['cfop_ipi'] != 'Natureza']

#### 3.3.4 Formatação dos dados

In [143]:
colunas_ipi_float = ['vl_ipi_contabilidade', 'vl_cont_ipi_contabilidade']
colunas_ipi_int = ['cfop_ipi', 'nf_ipi_contabilidade']
for col in colunas_ipi_int:
    df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('int')

for col in colunas_ipi_float:
    df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('float')

#### 3.3.5 Separando apenas os CFOP's menores ou iguais a 4000

In [144]:
filtro = df_contabilidade_ipi.query('cfop_ipi <= 4000')
df_contabilidade_ipi = filtro
df_contabilidade_ipi['id_unico_IPI'] = df_contabilidade_ipi['nf_ipi_contabilidade'].astype('str') + "|" + df_contabilidade_ipi['cfop_ipi'].astype('str')

#### 3.3.6 Criando a coluna ID unico

In [145]:
df_contabilidade_ipi['id_unico_ipi'] = df_contabilidade_ipi['nf_ipi_contabilidade'].astype(str) + '|' + df_contabilidade_ipi['cfop_ipi'].astype(str)

#### 3.3.7 Gerando os totais dos dados da contabilidade IPI

In [146]:
valor_ipi_total_contabilidade = df_contabilidade_ipi['vl_ipi_contabilidade'].sum()

## 4.0 Criando o dataframe com a lista mestre com todas os ID's unicos dos 3 dataframes

### 4.1 Lista mestre - Faz a inclusão de todos os Id's unicos e deixa apenas aparecendo uma unica vez removendo os duplicados

In [147]:
lista_mestre = pd.concat([
    df_empresa['id_unico_empresa'],
    df_contabilidade_icms['id_unico_ICMS'],
    df_contabilidade_ipi['id_unico_IPI']
]).dropna().drop_duplicates()

### 4.2 Faz a transformação da lista_mestre para um dataframe (df_mestre) para poder juntar com os outros dataframes

In [148]:
df_mestre = pd.DataFrame({'id_unico': lista_mestre})

### 4.3 Preenche os calores em branco com 0

In [149]:
df_mestre = df_mestre.fillna(0)

### 4.4 Acrescenta ao dataframe df_mestre as colunas de nf e cfop

In [150]:
df_mestre[['nf', 'cfop']] = df_mestre['id_unico'].str.split('|', expand=True)

## 5.0 União das Bases

### 5.1 Base Mestre + Base empresa

In [151]:
df_mestre_e_empresa = pd.merge(
    df_mestre,          # Sua Tabela 1
    df_empresa,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_empresa',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.2 (Base Mestre + Base empresa) + Base ICMS contabilidade

In [152]:
df_mestre_e_empresa_e_contabilidade_icms = pd.merge(
    df_mestre_e_empresa,          # Sua Tabela 1
    df_contabilidade_icms,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_ICMS',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.3 (Base Mestre + Base empresa + Base ICMS contabilidade + Base IPI contabilidade)

In [153]:
df_unido = pd.merge(
    df_mestre_e_empresa_e_contabilidade_icms,          # Sua Tabela 1
    df_contabilidade_ipi,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_IPI',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.4 Transformação dos dados

In [154]:
df_unido['cfop'] = pd.to_numeric(df_unido['cfop'], errors='coerce').astype('Int64')
df_unido['nf'] = pd.to_numeric(df_unido['nf'], errors='coerce').astype('Int64')
df_unido['nf_icms_contabilidade'] = pd.to_numeric(df_unido['nf_icms_contabilidade'], errors='coerce').astype('Int64')
df_unido['cfop_icms'] = pd.to_numeric(df_unido['cfop_icms'], errors='coerce').astype('Int64')

### 5.5 Criando as colunas de diferença e status

In [155]:
df_unido = df_unido.fillna(0)
df_unido['dif_vl_contabil'] = (df_unido['vl_cont_empresa'] - df_unido['vl_cont_icms_contabilidade']).round(2)
df_unido['dif_vl_icms'] = (df_unido['vl_icms_empresa'] - df_unido['vl_icms_contabilidade']).round(2)
df_unido['dif_vl_ipi'] = (df_unido['vl_ipi_empresa'] - df_unido['vl_ipi_contabilidade']).round(2)

cols_dif = ['dif_vl_contabil', 'dif_vl_icms', 'dif_vl_ipi','dif_vl_ipi']
df_unido['status'] = 'Valor Divergente'
df_unido.loc[df_unido[cols_dif].eq(0).all(axis=1), 'status'] = 'OK'
df_unido = df_unido.sort_values(by=['nf', 'cfop'])

### 5.6 Criando a base para exportação para o Excel

In [156]:
cols_desejadas = ['nf','cfop','vl_cont_empresa','vl_icms_empresa','vl_ipi_empresa', 'vl_cont_icms_contabilidade', 'vl_icms_contabilidade', 'vl_ipi_contabilidade', 'dif_vl_contabil', 'dif_vl_icms', 'dif_vl_ipi','status']
cols_existentes = [col for col in cols_desejadas if col in df_unido.columns]
df_export_excel = df_unido[cols_existentes]

## 6.0 DataFrame resumo

### 6.1 Criação do dataframe df_resumo e fazendo a exportação do arquivo em excel

In [157]:
df_resumo = pd.DataFrame({
    'Qtde Nfs Empresa': [qtde_nf_empresa],
    'Qtde Nfs Contabilidade': [qtde_nf_icms_contabilidade],
    'Diferença qtde Nfs': [qtde_nf_empresa - qtde_nf_icms_contabilidade],
    
    'Valor Total Contabil Empresa': [valor_contabil_total_empresa],
    'Valor Total Contabil Contabilidade': [valor_contabil_total_icms_contabilidade],
    'Diferença valor total Contabil': [valor_contabil_total_empresa - valor_contabil_total_icms_contabilidade],
    
    'Valor Total ICMS Empresa': [valor_icms_total_empresa],
    'Valor Total ICMS Contabilidade': [valor_icms_total_contabilidade],
    'Diferença Valor total ICMS': [valor_icms_total_empresa - valor_icms_total_contabilidade],
    
    'Valor Total IPI Empresa': [valor_ipi_total_empresa],
    'Valor Total IPI Contabilidade': [valor_ipi_total_contabilidade],
    'Diferença Valor total IPI': [valor_ipi_total_empresa - valor_ipi_total_contabilidade]
})
nome_arquivo = 'apuracao.xlsx'

with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
    df_export_excel.to_excel(writer, sheet_name='apuracao', index=False)
    df_resumo.to_excel(writer, sheet_name='resumo', index=False)


## 7.0 Formatação da Tabela em excel

In [158]:
# 2. Abre o arquivo salvo com o openpyxl para aplicar as cores de fundo
wb = load_workbook(nome_arquivo)
ws = wb.active

# Define as cores de preenchimento (Verde Claro e Vermelho Claro em formato Hexadecimal)
fill_verde = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid") # Verde claro (suave)
fill_vermelho = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid") # Vermelho claro (suave)

# Descobre qual é o número da coluna 'status'
coluna_status_idx = None
for col_idx, cell in enumerate(ws[1], 1):
    if cell.value == 'status':
        coluna_status_idx = col_idx
        break

# Se encontrou a coluna, percorre as linhas da tabela aplicando a cor correspondente
if coluna_status_idx:
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=coluna_status_idx)
        if cell.value == 'OK':
            cell.fill = fill_verde
        elif cell.value == 'Valor Divergente':
            cell.fill = fill_vermelho

# Salva o arquivo final com as cores aplicadas
wb.save(nome_arquivo)